In [5]:
# Importando Bibliotecas
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [6]:
# Fixa a semente para gerar sempre os mesmos dados fictícios
np.random.seed(42)

# 1. Gerando Massa de Dados de Servidores (30 dias)
datas_serv = pd.date_range(start='2026-08-01', periods=30, freq='D')
servidores = ['SRV-PROD-01', 'SRV-PROD-02', 'SRV-DB-01', 'SRV-APP-01']

data_serv = []
for d in datas_serv:
    for s in servidores:
        # Simulando pico de CPU aleatório (incluindo ruídos)
        cpu = np.random.normal(loc=65, scale=20)
        mem = np.random.normal(loc=70, scale=15)
        data_serv.append([d, s, round(cpu, 2), round(mem, 2)])

df_servidores = pd.DataFrame(data_serv, columns=['data_registro', 'servidor', 'cpu_pct', 'memoria_pct'])

# 2. Gerando Massa de Dados de Chamados de TI (100 registros)
datas_abertura = [datetime(2026, 8, 1) + timedelta(days=int(np.random.randint(0, 25))) for _ in range(100)]
data_chamados = []

for i, dt in enumerate(datas_abertura, 1):
    status = np.random.choice(['Resolvido', 'Aberto', 'Cancelado'], p=[0.7, 0.2, 0.1])
    prioridade = np.random.choice(['Alta', 'Média', 'Baixa'], p=[0.2, 0.5, 0.3])

    # Se resolvido, gera data de fechamento (de 1 a 72 horas depois)
    dt_fechamento = dt + timedelta(hours=int(np.random.randint(1, 72))) if status == 'Resolvido' else None

    data_chamados.append([f'INC-{1000+i}', dt, dt_fechamento, prioridade, status])

df_chamados = pd.DataFrame(data_chamados, columns=['id_chamado', 'data_abertura', 'data_fechamento', 'prioridade', 'status'])

print("Datasets gerados com sucesso!")

Datasets gerados com sucesso!


### Tratando dataset Chamados

In [13]:
df_chamados.head()

,id_chamado,data_abertura,data_fechamento,prioridade,status
0,INC-1001,2026-08-23,2026-08-24 12:00:00,Baixa,Resolvido
1,INC-1002,2026-08-05,2026-08-06 12:00:00,Baixa,Resolvido
2,INC-1003,2026-08-21,2026-08-23 16:00:00,Baixa,Resolvido
3,INC-1004,2026-08-23,2026-08-24 12:00:00,Baixa,Resolvido
4,INC-1005,2026-08-09,2026-08-10 00:00:00,Baixa,Resolvido


In [10]:
df_chamados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_chamado       100 non-null    object        
 1   data_abertura    100 non-null    datetime64[ns]
 2   data_fechamento  69 non-null     datetime64[ns]
 3   prioridade       100 non-null    object        
 4   status           100 non-null    object        
dtypes: datetime64[ns](2), object(3)
memory usage: 4.0+ KB


In [18]:
df_chamados.isnull().sum()

,0
id_chamado,0
data_abertura,0
data_fechamento,31
prioridade,0
status,0


In [19]:
df_chamados[df_chamados.isnull().any(axis=1)]

,id_chamado,data_abertura,data_fechamento,prioridade,status
5,INC-1006,2026-08-12,NaT,Média,Aberto
7,INC-1008,2026-08-01,NaT,Alta,Aberto
8,INC-1009,2026-08-01,NaT,Média,Cancelado
9,INC-1010,2026-08-15,NaT,Média,Aberto
10,INC-1011,2026-08-02,NaT,Média,Aberto
15,INC-1016,2026-08-13,NaT,Baixa,Aberto
17,INC-1018,2026-08-01,NaT,Alta,Cancelado
18,INC-1019,2026-08-16,NaT,Média,Aberto
19,INC-1020,2026-08-07,NaT,Baixa,Aberto
23,INC-1024,2026-08-03,NaT,Alta,Aberto


In [14]:
df_chamados.dtypes

,0
id_chamado,object
data_abertura,datetime64[ns]
data_fechamento,datetime64[ns]
prioridade,object
status,object


In [15]:
df_chamados.shape

(100, 5)

## Tratando o datset de servidores

In [20]:
# Exibe as 5 primeiras linhas dos servidores
df_servidores.head()

,data_registro,servidor,cpu_pct,memoria_pct
0,2026-08-01,SRV-PROD-01,74.93,67.93
1,2026-08-01,SRV-PROD-02,77.95,92.85
2,2026-08-01,SRV-DB-01,60.32,66.49
3,2026-08-01,SRV-APP-01,96.58,81.51
4,2026-08-02,SRV-PROD-01,55.61,78.14


In [21]:
df_servidores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   data_registro  120 non-null    datetime64[ns]
 1   servidor       120 non-null    object        
 2   cpu_pct        120 non-null    float64       
 3   memoria_pct    120 non-null    float64       
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 3.9+ KB


In [22]:
df_servidores.isnull().sum()

,0
data_registro,0
servidor,0
cpu_pct,0
memoria_pct,0


In [23]:
# Instlando o driver de conexão com SQL
!pip install pyodbc sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.1/350.1 kB 13.4 MB/s eta 0:00:00


In [26]:
# Exporta e baixa os CSVs diretamente para a sua máquina
df_servidores.to_csv('servidores_metrics.csv', index=False)
df_chamados.to_csv('chamados_ti.csv', index=False)

from google.colab import files
files.download('servidores_metrics.csv')
files.download('chamados_ti.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>